<a href="https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/BioEmu_Benchmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **BioEmu Benchmarks**

This notebook runs the [BioEmu benchmarks](https://github.com/microsoft/bioemu-benchmarks) on `topology.pdb` + `samples.xtc` outputs from the [BioEmu Colab notebook](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/BioEmu.ipynb).

**Runtime:** No GPU required. All benchmarks run on CPU only — a standard (free) Colab runtime is sufficient.

## Available Benchmarks

| Benchmark | Description | Recommended Samples |
|-----------|-------------|--------------------|
| `multiconf_ood60` | Local conformational changes (out-of-distribution) | 4,000/protein |
| `multiconf_oodval` | Global conformational changes (validation) | 4,000/protein |
| `multiconf_domainmotion` | Global domain motions | 4,000/protein |
| `multiconf_crypticpocket` | Cryptic pocket backbone changes | 4,000/protein |
| `singleconf_localunfolding` | Local protein unfolding | 4,000/protein |
| `folding_free_energies` | Folding free energy prediction | 200-7,600/protein |
| `md_emulation` | MD distribution matching | 10,000/protein |

## Input Format

Upload your BioEmu samples organized as:
```
my_samples/
├── protein_1/
│   ├── topology.pdb
│   └── samples.xtc
├── protein_2/
│   ├── topology.pdb
│   └── samples.xtc
└── ...
```
Each benchmark requires samples for specific protein sequences. Use the **"Show benchmark specs"** cell to see which sequences are needed.

In [ ]:
#@title Install dependencies
import os
import sys

_is_bench_setup_file = '/content/.BIOEMU_BENCH_SETUP'

if not os.path.exists(_is_bench_setup_file):
    # Install bioemu-benchmarks from fork with fuzzy matching and custom protein support
    os.system('pip install -q "bioemu-benchmarks @ git+https://github.com/ts387/bioemu-benchmarks.git@claude/fix-multiconf-sample-matching-ov4lD"')
    os.system(f'touch {_is_bench_setup_file}')
    print('Installation complete.')
else:
    print('Dependencies already installed.')

In [ ]:
#@title Configure sample directory and output
#@markdown - `samples_dir`: Path to directory containing your BioEmu samples (topology.pdb + samples.xtc pairs)
#@markdown - `output_dir`: Path where benchmark results will be saved
#@markdown - `filter_samples`: Filter out unphysical samples (chain breaks, clashes) before evaluation
#@markdown - `use_google_drive`: Mount Google Drive for input/output

samples_dir = "/content/my_samples" #@param {type:"string"}
output_dir = "/content/benchmark_results" #@param {type:"string"}
filter_samples = True #@param {type:"boolean"}
use_google_drive = False #@param {type:"boolean"}

if use_google_drive:
    from google.colab import drive
    drive.mount('/content/drive')

os.makedirs(output_dir, exist_ok=True)
print(f'Samples directory: {samples_dir}')
print(f'Output directory: {output_dir}')

In [ ]:
#@title Show benchmark specs (sequences & recommended sample sizes)
#@markdown Select a benchmark to see its required sequences and recommended sample counts.

benchmark_to_show = "multiconf_ood60" #@param ["multiconf_ood60", "multiconf_oodval", "multiconf_domainmotion", "multiconf_crypticpocket", "singleconf_localunfolding", "folding_free_energies", "md_emulation"]

from bioemu_benchmarks.benchmarks import Benchmark

bm = Benchmark(benchmark_to_show)
metadata = bm.metadata.copy()
metadata['recommended_samples'] = bm.default_samplesize
print(f'Benchmark: {benchmark_to_show}')
print(f'Number of test cases: {len(metadata)}')
print(f'Number of unique sequences: {metadata["sequence"].nunique()}')
print()
display(metadata)

In [ ]:
#@title Load and validate samples
#@markdown This cell discovers all topology.pdb + samples.xtc pairs in your samples directory
#@markdown and reports what was found. Corrupt or truncated XTC files are flagged with guidance.

import mdtraj
from bioemu_benchmarks.samples import find_samples_in_dir

sequence_samples = find_samples_in_dir(samples_dir)
print(f'Found {len(sequence_samples)} sample file(s):')
print()

_valid_samples = []
_failed_samples = []

for ss in sequence_samples:
    try:
        top = mdtraj.load_topology(ss.topology_file)
        seq = top.to_fasta()[0]
        traj = mdtraj.load(ss.trajectory_file, top=ss.topology_file)
        print(f'  {ss.trajectory_file}')
        print(f'    Sequence ({len(seq)} residues): {seq[:50]}{"..." if len(seq) > 50 else ""}')
        print(f'    Frames: {traj.n_frames}')
        print()
        _valid_samples.append(ss)
    except (RuntimeError, OSError, ValueError) as e:
        import os
        xtc_size = os.path.getsize(ss.trajectory_file)
        print(f'  {ss.trajectory_file}')
        print(f'    ERROR: {e}')
        print(f'    File size: {xtc_size / 1024:.1f} KB')
        print(f'    This XTC file is likely corrupt or truncated (incomplete write).')
        print(f'    Common causes: Colab runtime timeout, out-of-memory, or disk full')
        print(f'    during BioEmu sampling. Re-run sampling for this protein.')
        print()
        _failed_samples.append(ss)

if _failed_samples:
    print(f'WARNING: {len(_failed_samples)}/{len(sequence_samples)} sample file(s) '
          f'failed to load and will be skipped by benchmarks.')
    # Update sequence_samples so downstream cells only use valid files
    sequence_samples = _valid_samples
elif sequence_samples:
    print(f'All {len(sequence_samples)} sample file(s) loaded successfully.')

In [ ]:
#@title Helper: run a single benchmark

import json
import warnings
from pathlib import Path

from bioemu_benchmarks.benchmarks import Benchmark
from bioemu_benchmarks.evaluator_utils import evaluator_from_benchmark
from bioemu_benchmarks.samples import (
    IndexedSamples,
    NoSamples,
    filter_unphysical_samples,
    find_samples_in_dir,
)


def run_benchmark(benchmark_name, samples_dir, output_dir, filter_samples=True):
    """Run a single benchmark and return the results object."""
    benchmark = Benchmark(benchmark_name)
    results_dir = Path(output_dir) / benchmark_name
    results_dir.mkdir(parents=True, exist_ok=True)

    # Load samples
    sequence_samples = find_samples_in_dir(samples_dir)

    try:
        indexed_samples = IndexedSamples.from_benchmark(
            benchmark=benchmark, sequence_samples=sequence_samples
        )
    except NoSamples:
        print(f'No matching samples found for {benchmark_name}. '
              f'Use the "Show benchmark specs" cell to see required sequences.')
        return None

    matched_cases = list(indexed_samples.test_case_to_sequencesamples.keys())
    total_cases = len(benchmark.metadata)
    print(f'Matched {len(matched_cases)}/{total_cases} test cases')

    # Filter unphysical samples
    if filter_samples:
        print('Filtering unphysical samples...')
        indexed_samples, filter_stats = filter_unphysical_samples(indexed_samples)
        filter_stats_mean = {k: float(v.mean()) for k, v in filter_stats.items()}
        with open(results_dir / 'filter_statistics.json', 'w') as f:
            json.dump(filter_stats_mean, f, indent=2, sort_keys=True)
        avg_kept = sum(filter_stats_mean.values()) / len(filter_stats_mean) if filter_stats_mean else 0
        print(f'Average fraction of samples kept after filtering: {avg_kept:.2%}')

    # Run evaluation
    print(f'Running {benchmark_name} evaluation...')
    evaluator = evaluator_from_benchmark(benchmark=benchmark)
    results = evaluator(indexed_samples)

    # Save results
    print('Saving results...')
    results.save_results(results_dir)
    results.to_pickle(results_dir / 'results.pkl')

    # Plot
    print('Generating plots...')
    results.plot(results_dir)

    # Aggregate metrics
    aggregate = results.get_aggregate_metrics()
    with open(results_dir / 'aggregate_metrics.json', 'w') as f:
        json.dump(aggregate, f, indent=2, sort_keys=True)

    print(f'\nResults saved to: {results_dir}')
    print(f'\nAggregate metrics:')
    for k, v in aggregate.items():
        print(f'  {k}: {v:.4f}')

    return results

print('Helper loaded.')

---
## Run Individual Benchmarks
Run only the benchmarks for which you have matching samples. Each cell is independent.

---
## Custom Protein Analysis
Evaluate BioEmu samples for **any protein** — no benchmark membership required.
Computes ensemble quality metrics and structural comparison against a reference PDB
(e.g., AlphaFold3 prediction, experimental structure, or topology.pdb as fallback).

**Metrics computed:**
- **Ensemble quality** (no reference needed): physical validity rate, pairwise RMSD diversity, radius of gyration
- **Reference comparison**: RMSD, TM-score, lDDT, contact distance, DSSP accuracy

In [ ]:
#@title Custom protein analysis
#@markdown Evaluate BioEmu samples for **any protein** against a reference structure.
#@markdown
#@markdown - `reference_pdb`: Path to a reference PDB (e.g., AlphaFold3 prediction, experimental structure).
#@markdown   Leave empty to use each sample's `topology.pdb` as its own reference.
#@markdown - `filter_samples`: Uses the same setting from the config cell above.

reference_pdb = "" #@param {type:"string"}

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mdtraj
import numpy as np

from bioemu_benchmarks.eval.multiconf.evaluate import MetricType, get_metrics_against_references
from bioemu_benchmarks.samples import find_samples_in_dir
from bioemu_benchmarks.utils import get_physical_traj_indices

custom_results_dir = Path(output_dir) / 'custom_analysis'
custom_results_dir.mkdir(parents=True, exist_ok=True)

# Discover samples
_custom_samples = find_samples_in_dir(samples_dir)
if not _custom_samples:
    raise RuntimeError(f'No samples found in {samples_dir}. '
                       f'Check that your directory contains topology.pdb + samples.xtc pairs.')

print(f'Found {len(_custom_samples)} sample set(s)\n')

# Load shared reference if provided
_shared_ref = None
if reference_pdb.strip():
    _shared_ref = mdtraj.load_pdb(reference_pdb.strip())
    print(f'Reference: {reference_pdb} ({_shared_ref.topology.n_residues} residues)\n')
else:
    print('No reference PDB provided — using each topology.pdb as its own reference.\n')

_metric_types = [MetricType.RMSD, MetricType.TMSCORE, MetricType.LDDT,
                 MetricType.CONTACT_DISTANCE, MetricType.DSSP_ACC]

_all_custom_results = {}

for _ss in _custom_samples:
    _label = Path(_ss.trajectory_file).stem
    _parent = Path(_ss.trajectory_file).parent.name
    _name = f'{_parent}/{_label}' if _parent != Path(samples_dir).name else _label
    print(f'{"=" * 60}')
    print(f'Protein: {_name}')
    print(f'{"=" * 60}')

    # Load trajectory
    _traj = mdtraj.load(_ss.trajectory_file, top=_ss.topology_file)
    _seq = _traj.topology.to_fasta()[0]
    print(f'  Sequence ({len(_seq)} residues): {_seq[:50]}{"..." if len(_seq) > 50 else ""}')
    print(f'  Samples: {_traj.n_frames}')

    # --- Ensemble quality (no reference needed) ---

    # Physical validity
    _valid_idx = get_physical_traj_indices(_traj)
    _validity_rate = len(_valid_idx) / _traj.n_frames
    print(f'  Physical validity: {_validity_rate:.1%} ({len(_valid_idx)}/{_traj.n_frames})')

    # Filter if requested
    if filter_samples and len(_valid_idx) > 0:
        _traj_eval = _traj[_valid_idx]
        print(f'  Using {_traj_eval.n_frames} physically valid samples for evaluation')
    elif len(_valid_idx) == 0:
        print(f'  WARNING: No physically valid samples. Using all {_traj.n_frames} unfiltered.')
        _traj_eval = _traj
    else:
        _traj_eval = _traj

    # Radius of gyration
    _rg = mdtraj.compute_rg(_traj_eval) * 10  # nm -> Angstrom
    print(f'  Radius of gyration: {np.mean(_rg):.1f} +/- {np.std(_rg):.1f} A')

    # Pairwise RMSD (subsample for speed)
    _n_pw = min(200, _traj_eval.n_frames)
    _pw_idx = np.random.choice(_traj_eval.n_frames, _n_pw, replace=False) if _traj_eval.n_frames > _n_pw else np.arange(_traj_eval.n_frames)
    _bb_traj = _traj_eval.atom_slice(_traj_eval.topology.select('backbone'))
    _pw_rmsd = np.zeros((_n_pw * (_n_pw - 1)) // 2)
    _k = 0
    for _i in range(_n_pw):
        for _j in range(_i + 1, _n_pw):
            _pw_rmsd[_k] = mdtraj.rmsd(_bb_traj[_pw_idx[_i]], _bb_traj, frame=_pw_idx[_j])[0]
            _k += 1
    _pw_rmsd *= 10  # nm -> Angstrom
    print(f'  Pairwise RMSD (diversity): {np.mean(_pw_rmsd):.2f} +/- {np.std(_pw_rmsd):.2f} A')

    # --- Reference comparison ---

    _ref = _shared_ref if _shared_ref is not None else mdtraj.load_pdb(str(_ss.topology_file))
    _ref_label = reference_pdb.strip() if _shared_ref is not None else 'topology.pdb'
    print(f'  Reference: {_ref_label} ({_ref.topology.n_residues} residues)')

    try:
        _metrics = get_metrics_against_references(
            sample_traj=_traj_eval,
            ref_trajs=[_ref],
            metric_types=_metric_types,
        )

        _result = {}
        print(f'\n  Metrics vs reference (mean +/- std):')
        for _mt, _vals in _metrics.items():
            _v = _vals[:, 0]  # single reference -> column 0
            _result[_mt.value] = {'mean': float(np.mean(_v)), 'std': float(np.std(_v)),
                                  'median': float(np.median(_v))}
            if _mt == MetricType.RMSD:
                _v_display = _v * 10  # nm -> Angstrom
                print(f'    {_mt.value}: {np.mean(_v_display):.2f} +/- {np.std(_v_display):.2f} A '
                      f'(min: {np.min(_v_display):.2f}, median: {np.median(_v_display):.2f})')
            else:
                print(f'    {_mt.value}: {np.mean(_v):.3f} +/- {np.std(_v):.3f} '
                      f'(min: {np.min(_v):.3f}, median: {np.median(_v):.3f})')
        _result['validity_rate'] = _validity_rate
        _result['n_samples'] = _traj.n_frames
        _result['n_valid'] = len(_valid_idx)
        _result['rg_mean'] = float(np.mean(_rg))
        _result['pw_rmsd_mean'] = float(np.mean(_pw_rmsd))
        _all_custom_results[_name] = _result

        # --- Plots ---
        _fig, _axes = plt.subplots(2, 3, figsize=(15, 8))
        _fig.suptitle(f'{_name}', fontsize=14)

        for _ax, (_mt, _vals) in zip(_axes.flat, _metrics.items()):
            _v = _vals[:, 0]
            _unit = ' (A)' if _mt == MetricType.RMSD else ''
            if _mt == MetricType.RMSD:
                _v = _v * 10
            _ax.hist(_v, bins=50, alpha=0.7, edgecolor='black', linewidth=0.5)
            _ax.set_xlabel(f'{_mt.value}{_unit}')
            _ax.set_ylabel('Count')
            _ax.axvline(np.mean(_v), color='red', linestyle='--', label=f'mean={np.mean(_v):.2f}')
            _ax.legend(fontsize=8)

        # Rg in last subplot
        _axes[1, 2].hist(_rg, bins=50, alpha=0.7, color='green', edgecolor='black', linewidth=0.5)
        _axes[1, 2].set_xlabel('Radius of gyration (A)')
        _axes[1, 2].set_ylabel('Count')
        _axes[1, 2].axvline(np.mean(_rg), color='red', linestyle='--', label=f'mean={np.mean(_rg):.1f}')
        _axes[1, 2].legend(fontsize=8)

        plt.tight_layout()
        _plot_path = custom_results_dir / f'{_name.replace("/", "_")}_metrics.png'
        plt.savefig(_plot_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'  Plot saved: {_plot_path}\n')

    except Exception as _e:
        print(f'  ERROR computing metrics: {_e}')
        print(f'  This may happen if sample and reference sequences are too different.\n')

# Save aggregate results
if _all_custom_results:
    _results_path = custom_results_dir / 'custom_metrics.json'
    with open(_results_path, 'w') as _f:
        json.dump(_all_custom_results, _f, indent=2, sort_keys=True)
    print(f'\nResults saved to: {_results_path}')

In [ ]:
#@title Multiconf OOD60
#@markdown Evaluates local conformational changes on out-of-distribution proteins.
#@markdown Metrics: RMSD, TM-score, lDDT, contact distance, DSSP accuracy.
#@markdown Recommended: 4,000 samples per protein, 22 test cases.

results_ood60 = run_benchmark('multiconf_ood60', samples_dir, output_dir, filter_samples)

In [ ]:
#@title Multiconf OODVAL
#@markdown Evaluates global conformational changes on validation proteins.
#@markdown Metrics: RMSD, TM-score, lDDT, contact distance, DSSP accuracy.
#@markdown Recommended: 4,000 samples per protein, 18 test cases.

results_oodval = run_benchmark('multiconf_oodval', samples_dir, output_dir, filter_samples)

In [ ]:
#@title Multiconf Domain Motion
#@markdown Evaluates global protein domain motions.
#@markdown Metrics: RMSD, TM-score, lDDT, contact distance, DSSP accuracy.
#@markdown Recommended: 4,000 samples per protein, 27 test cases.

results_domainmotion = run_benchmark('multiconf_domainmotion', samples_dir, output_dir, filter_samples)

In [ ]:
#@title Multiconf Cryptic Pocket
#@markdown Evaluates pocket backbone changes upon ligand binding (holo vs apo).
#@markdown Metrics: RMSD, TM-score, lDDT, contact distance, DSSP accuracy.
#@markdown Recommended: 4,000 samples per protein, 34 test cases.

results_crypticpocket = run_benchmark('multiconf_crypticpocket', samples_dir, output_dir, filter_samples)

In [ ]:
#@title Singleconf Local Unfolding
#@markdown Evaluates local protein unfolding via fraction of native contacts.
#@markdown Metrics: FNC (fold/unfold regions).
#@markdown Recommended: 4,000 samples per protein, 20 test cases.

results_localunfolding = run_benchmark('singleconf_localunfolding', samples_dir, output_dir, filter_samples)

In [ ]:
#@title Folding Free Energies
#@markdown Predicts folding free energies (dG and ddG) from sample ensembles.
#@markdown Metrics: MAE and correlation for dG and ddG vs experiment.
#@markdown Recommended: 200-7,600 samples per protein, 364 test cases (~100+ unique sequences).

results_folding = run_benchmark('folding_free_energies', samples_dir, output_dir, filter_samples)

In [ ]:
#@title MD Emulation
#@markdown Compares sample distributions to MD reference on projected free energy surfaces.
#@markdown Metrics: Free energy MAE, RMSE, and coverage.
#@markdown Recommended: 10,000 samples per protein, 16 test cases.

results_md = run_benchmark('md_emulation', samples_dir, output_dir, filter_samples)

---
## Run All Benchmarks at Once
Alternatively, run all benchmarks in sequence. Only benchmarks with matching samples will produce results.

In [ ]:
#@title Run all benchmarks

all_benchmark_names = [
    'multiconf_ood60',
    'multiconf_oodval',
    'multiconf_domainmotion',
    'multiconf_crypticpocket',
    'singleconf_localunfolding',
    'folding_free_energies',
    'md_emulation',
]

all_results = {}
all_aggregate = {}

for bm_name in all_benchmark_names:
    print(f'\n{"=" * 60}')
    print(f'Running: {bm_name}')
    print(f'{"=" * 60}')
    result = run_benchmark(bm_name, samples_dir, output_dir, filter_samples)
    if result is not None:
        all_results[bm_name] = result
        all_aggregate[bm_name] = result.get_aggregate_metrics()

# Save combined aggregate metrics
combined_path = Path(output_dir) / 'benchmark_metrics.json'
with open(combined_path, 'w') as f:
    json.dump(all_aggregate, f, indent=2, sort_keys=True)

print(f'\n{"=" * 60}')
print(f'All benchmarks complete. Combined metrics saved to: {combined_path}')
print(f'{"=" * 60}')

In [ ]:
#@title Display generated plots
#@markdown Show plots from all completed benchmarks.

from IPython.display import display, Image
from pathlib import Path

results_path = Path(output_dir)
png_files = sorted(results_path.glob('**/*.png'))

if not png_files:
    print('No plots found. Run a benchmark first.')
else:
    for png in png_files:
        relative = png.relative_to(results_path)
        print(f'\n--- {relative} ---')
        display(Image(filename=str(png), width=600))